# GraphRAG 구축 + Advanced 벡터RAG 비교

1. Microsoft GraphRAG로 `reports.json` 인덱싱 (엔티티 추출 → 커뮤니티 탐지 → 요약)
2. Local Search / Global Search 실습
3. Advanced 벡터RAG와 동일 질의로 응답 비교

## 0. 환경 설정 및 패키지 설치


In [25]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports_structured.json
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports_small.json

--2026-06-24 01:07:06--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports_structured.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 140040 (137K) [text/plain]
Saving to: ‘reports_structured.json.8’

reports_structured. 100%[===================>] 136.76K  --.-KB/s    in 0.003s  

2026-06-24 01:07:06 (50.0 MB/s) - ‘reports_structured.json.8’ saved [140040/140040]

--2026-06-24 01:07:06--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports_small.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Leng

In [ ]:
!pip install -q --force-reinstall "graphrag==1.0.1"
!pip install -q pandas pyyaml ragas datasets langchain-ollama
!pip install numpy==1.26.4

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
instructor 1.15.3 requires jiter<0.15,>=0.6.1, but you have jiter 0.15.0 which is incompatible.
instructor 1.15.3 requires openai<3.0.0,>=2.0.0, but you have openai 1.109.1 which is incompatible.
langchain-openai 1.3.3 requires openai<3.0.0,>=2.26.0, but you have openai 1.109.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
cuml

In [27]:
!apt-get -qq update
!apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [28]:
import json
import os
import subprocess
import time

with open("reports_small.json", "r", encoding="utf-8") as f:
    reports = json.load(f)

reports = reports[:40]
print(f"총 {len(reports)}건 로드")


총 40건 로드


In [29]:
# Ollama 서버 확인 (NB-03에서 이미 띄웠다면 재시작 불필요)
import requests
try:
    requests.get("http://localhost:11434/api/tags", timeout=3)
    print("Ollama 서버 이미 실행 중")
except Exception:
    process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(5)
    print("Ollama 서버 새로 시작됨")


Ollama 서버 이미 실행 중


In [30]:
!pkill -9 ollama

import os, subprocess, time
os.environ["OLLAMA_KEEP_ALIVE"] = "-1"  # 무제한 유지, Ollama+GraphRAG 연동 시 OLLAMA_KEEP_ALIVE를 길게 설정
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=os.environ)
time.sleep(5)

- 한글용 임베딩 
    - nomic-embed-text: nomic ai에서 개발, 768차원, 최대토큰 8192, 영어 중심이지만 multilingual지원, 경량
    - multilingual-e5-large: 100개 언어, 한글 우수
    - bge-m3 : 한중일특화, 성능 우수, 실제 서비스 활용 가능
    - paraphrase-multilingual-mpnet: 경량, 무난

In [31]:
#!ollama pull gemma:2b
!ollama pull nomic-embed-text
#!ollama pull llama3:8b-instruct-q4_0

In [32]:
!ollama list

NAME                              ID              SIZE      MODIFIED               
nomic-embed-text:latest           0a109f422b47    274 MB    Less than a second ago    
llama3:8b-instruct-q4_0           365c0bd3c000    4.7 GB    2 hours ago               
gemma:2b                          b50d6c999e59    1.7 GB    2 hours ago               
engine-diagnosis-expert:latest    d41b1ccf5af8    2.0 GB    3 hours ago               
qwen2.5:3b                        357c53fb659c    1.9 GB    3 hours ago               
llama3.2:3b                       a80c4f17acd5    2.0 GB    3 hours ago               
mistral:7b-instruct-q2_K          1344ecf13c2e    3.1 GB    3 hours ago               


## 1. GraphRAG 프로젝트 구조 초기화

GraphRAG는 별도 디렉토리 구조(`input/`, `output/`, `settings.yaml`)를 요구합니다.


In [33]:
!rm -r graphrag_project/

In [34]:
GRAPHRAG_ROOT = "./graphrag_project"

os.makedirs(f"{GRAPHRAG_ROOT}/input", exist_ok=True)

# 보고서 텍스트를 GraphRAG 입력 형식(개별 txt 파일)으로 저장
for r in reports:  
    file_path = f"{GRAPHRAG_ROOT}/input/{r['report_id']}.txt"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(r["report_text"])

print(f"{len(reports)}개 입력 파일 생성 완료: {GRAPHRAG_ROOT}/input/")


40개 입력 파일 생성 완료: ./graphrag_project/input/


In [35]:

#!cd {GRAPHRAG_ROOT} && python -m graphrag init --root .
!cd {GRAPHRAG_ROOT} && printf '\n\n\n\n\n\n\n\n' | python -m graphrag init --root .         #엔터 친 것과 같은 효과


Initializing project at /content/graphrag_project
⠋ GraphRAG Indexer 

In [36]:
import os

os.makedirs(f"{GRAPHRAG_ROOT}/input", exist_ok=True)
os.makedirs(f"{GRAPHRAG_ROOT}/output", exist_ok=True)
os.makedirs(f"{GRAPHRAG_ROOT}/prompts", exist_ok=True)  # graphrag가 기본 프롬프트 템플릿을 요구할 수 있어 미리 생성

print("GraphRAG 디렉토리 구조 생성 완료 - init 건너뜀")

GraphRAG 디렉토리 구조 생성 완료 - init 건너뜀


## 2. settings.yaml 설정: Ollama 로컬 LLM 연결

GraphRAG는 기본적으로 OpenAI API를 가정하므로, Ollama를 OpenAI 호환 엔드포인트로 연결하도록 설정을 수정합니다.


In [37]:
# 1. .env 파일 생성
env_content = '''GRAPHRAG_API_KEY=ollama
'''
with open(f"{GRAPHRAG_ROOT}/.env", "w", encoding="utf-8") as f:
    f.write(env_content)

print(".env 파일 생성 완료")

.env 파일 생성 완료


In [38]:
# 2. settings.yaml에서 api_key를 환경변수 참조 방식으로 수정
settings_yaml = '''llm:
  api_key: ${GRAPHRAG_API_KEY}
  type: openai_chat
  model: qwen2.5:3b
  model_supports_json: false
  max_tokens: 4000
  api_base: http://localhost:11434/v1
  concurrent_requests: 1

embeddings:
  llm:
    api_key: ${GRAPHRAG_API_KEY}
    type: openai_embedding
    model: nomic-embed-text
    api_base: http://localhost:11434/v1
    concurrent_requests: 1
  vector_store:
    type: lancedb
    db_uri: "output/lancedb"
    container_name: default
    overwrite: true

input:
  type: file
  file_type: text
  base_dir: "input"
  file_pattern: ".*\\\\.txt"

chunks:
  size: 300
  overlap: 50

entity_extraction:
  entity_types: [설비, 부품, 불량유형, 원인, 조치]
  max_gleanings: 1

community_reports:
  max_length: 1500

output:
  type: file
  base_dir: "output"
'''

with open(f"{GRAPHRAG_ROOT}/settings.yaml", "w", encoding="utf-8") as f:
    f.write(settings_yaml)

print(settings_yaml)

llm:
  api_key: ${GRAPHRAG_API_KEY}
  type: openai_chat
  model: qwen2.5:3b
  model_supports_json: false
  max_tokens: 4000
  api_base: http://localhost:11434/v1
  concurrent_requests: 1

embeddings:
  llm:
    api_key: ${GRAPHRAG_API_KEY}
    type: openai_embedding
    model: nomic-embed-text
    api_base: http://localhost:11434/v1
    concurrent_requests: 1
  vector_store:
    type: lancedb
    db_uri: "output/lancedb"
    container_name: default
    overwrite: true

input:
  type: file
  file_type: text
  base_dir: "input"
  file_pattern: ".*\\.txt"

chunks:
  size: 300
  overlap: 50

entity_extraction:
  entity_types: [설비, 부품, 불량유형, 원인, 조치]
  max_gleanings: 1

community_reports:
  max_length: 1500

output:
  type: file
  base_dir: "output"



In [39]:
# Colab에서 실행 중인 셀 정지 버튼으로 중단한 뒤
!pkill ollama
import subprocess, time
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

## 3. GraphRAG 인덱싱 실행

엔티티/관계 추출 → 커뮤니티 탐지(Leiden) → 커뮤니티별 요약까지 한 번에 수행됩니다.


In [40]:
import os
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"
os.environ["OPENAI_API_KEY"] = "ollama"

# 확인
print(os.environ.get("OPENAI_BASE_URL"))
print(os.environ.get("OPENAI_API_KEY"))

http://localhost:11434/v1
ollama


- ollama 재시작 필요 시

In [41]:
!pkill -9 ollama
import os, subprocess, time

os.environ["OLLAMA_KEEP_ALIVE"] = "-1"  # 무제한 유지, Ollama+GraphRAG 연동 시 OLLAMA_KEEP_ALIVE를 길게 설정

process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, env=os.environ)
time.sleep(5)


In [42]:
!ollama list
!curl -s --max-time 5 http://localhost:11434/api/tags

NAME                              ID              SIZE      MODIFIED       
nomic-embed-text:latest           0a109f422b47    274 MB    15 seconds ago    
llama3:8b-instruct-q4_0           365c0bd3c000    4.7 GB    2 hours ago       
gemma:2b                          b50d6c999e59    1.7 GB    2 hours ago       
engine-diagnosis-expert:latest    d41b1ccf5af8    2.0 GB    3 hours ago       
qwen2.5:3b                        357c53fb659c    1.9 GB    3 hours ago       
llama3.2:3b                       a80c4f17acd5    2.0 GB    3 hours ago       
mistral:7b-instruct-q2_K          1344ecf13c2e    3.1 GB    3 hours ago       
{"models":[{"name":"nomic-embed-text:latest","model":"nomic-embed-text:latest","modified_at":"2026-06-24T01:09:04.718785693Z","size":274302450,"digest":"0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f","details":{"parent_model":"","format":"gguf","family":"nomic-bert","families":["nomic-bert"],"parameter_size":"137M","quantization_level":"F16","context

In [43]:
import requests, time

# nomic-embed-text를 미리 한 번 호출해서 GPU에 로드해두기
print("임베딩 모델 워밍업 중...")
response = requests.post(
    "http://localhost:11434/v1/embeddings",
    json={"model": "nomic-embed-text", "input": "워밍업 테스트"},
    timeout=120,
)
print("상태 코드:", response.status_code)
print("워밍업 완료, 인덱싱 시작")

임베딩 모델 워밍업 중...
상태 코드: 200
워밍업 완료, 인덱싱 시작


In [44]:
GRAPHRAG_ROOT = "./graphrag_project"  #서버 재시작한 경우

In [45]:
# 시간이 오래 걸리는 셀
#!rm -rf {GRAPHRAG_ROOT}/output {GRAPHRAG_ROOT}/cache       #재인덱싱하는 경우
!cd {GRAPHRAG_ROOT} && python -m graphrag index --root .



Logging enabled at /content/graphrag_project/logs/indexing-engine.log
⠹ GraphRAG Indexer 
⠹ GraphRAG Indexer e.text) - 40 files loaded (0 filtered) ━ 100%  0…
├── Loading Input (InputFileType.text) - 40 files loaded (0 filtered) ━ 100%  0…
⠹ GraphRAG Indexer 
├── Loading Input (InputFileType.text) - 40 files loaded (0 filtered) ━ 100%  0…
⠋ GraphRAG Indexer 
├── Loading Input (InputFileType.text) - 40 files loaded (0 filtered) ━ 100%  0…
🚀 create_base_text_units
⠋ GraphRAG Indexer 
├── Loading Input (InputFileType.text) - 40 files loaded (0 filtered) ━ 100%  0…
Empty DataFrame
Columns: []
Index: []
⠋ GraphRAG Indexer 
├── Loading Input (InputFileType.text) - 40 files loaded (0 filtered) ━ 100%  0…
⠹ GraphRAG Indexer 
├── Loading Input (InputFileType.text) - 40 files loaded (0 filtered) ━ 100%  0…
⠹ GraphRAG Indexer 
├── Loading Input (InputFileType.text) - 40 files loaded (0 filtered) ━ 100%  0…
├── create_base_text_units
🚀 create_final_documents
⠸ GraphRAG Indexer 
├── Loading Input 

In [46]:
!ls graphrag_project/output

create_final_communities.parquet	create_final_relationships.parquet
create_final_community_reports.parquet	create_final_text_units.parquet
create_final_documents.parquet		lancedb
create_final_entities.parquet		stats.json
create_final_nodes.parquet


In [51]:
import pandas as pd

entities_df = pd.read_parquet(f"{GRAPHRAG_ROOT}/output/create_final_entities.parquet")
relationships_df = pd.read_parquet(f"{GRAPHRAG_ROOT}/output/create_final_relationships.parquet")
communities_df = pd.read_parquet(f"{GRAPHRAG_ROOT}/output/create_final_communities.parquet")

print(f"추출된 엔티티 수: {len(entities_df)}")
print(f"추출된 관계 수: {len(relationships_df)}")
print(f"탐지된 커뮤니티 수: {len(communities_df)}")

entities_df[["title", "type", "description"]].head(10)


추출된 엔티티 수: 218
추출된 관계 수: 201
탐지된 커뮤니티 수: 4


,title,type,description
0,체결 볼트 2개,부품,"两个紧固螺栓，用于将零件固定在基座上\n두 개의 체결 볼트,用于将零件固定在基座上"
1,불량유형: 낮은 토크 체결,불량유형,螺栓扭矩不足导致的不良部件
2,원인: 작업자 토크 설정 오류,원인,工人在紧固过程中错误地使用了之前的设定值，导致扭矩不足
3,품질관리팀,부품,"""품질관리팀""负责检查和分类不合格的部件，并在Snap-fit组装后使用间隙尺进行检查，以确..."
4,조치: 재체결 및 교육,,
5,절차를 재교육함,조치,Re-training the procedure is performed)<br>
6,작업지시서 첫 화면에 품목별 체결 토크 확인 체크 항목을 추가함,설비,An additional check item for item-specific tig...
7,CNC 머신 D-1호기,설비,This is a CNC machine used for manufacturing
8,브래킷 슬롯 길이,부품,"The length of the bracket slot, which is part ..."
9,도면 42.00MM,불량유형,The standard dimension for the bracket slot as...


## 4. Local Search / Global Search 실습


In [52]:
# Local Search: 특정 엔티티 중심 세부 질의
local_query = "CNC 가공기 실린더블록용 A-3호기에서 발생한 불량의 원인은?"

!cd {GRAPHRAG_ROOT} && python -m graphrag query \
    --root . \
    --method local \
    --query "{local_query}"





INFO: Vector Store Args: {
    "type": "lancedb",
    "db_uri": "/content/graphrag_project/output/lancedb",
    "container_name": "==== REDACTED ====",
    "overwrite": true
}
creating llm client with {'api_key': 'REDACTED,len=6', 'type': "openai_chat", 'encoding_model': 'cl100k_base', 'model': 'qwen2.5:3b', 'max_tokens': 4000, 'temperature': 0.0, 'top_p': 1.0, 'n': 1, 'frequency_penalty': 0.0, 'presence_penalty': 0.0, 'request_timeout': 180.0, 'api_base': 'http://localhost:11434/v1', 'api_version': None, 'organization': None, 'proxy': None, 'audience': None, 'deployment_name': None, 'model_supports_json': False, 'tokens_per_minute': 0, 'requests_per_minute': 0, 'max_retries': 10, 'max_retry_wait': 10.0, 'sleep_on_rate_limit_recommendation': True, 'concurrent_requests': 1, 'responses': None}
creating embedding llm client with {'api_key': 'REDACTED,len=6', 'type': "openai_embedding", 'encoding_model': 'cl100k_base', 'model': 'nomic-embed-text', 'max_tokens': 4000, 'temperature': 0, '

In [53]:
# Global Search: 전체 문서셋의 주제·패턴 파악
global_query = "전체 보고서에서 가장 빈번하게 나타나는 4M 원인 유형은 무엇인가?"

!cd {GRAPHRAG_ROOT} && python -m graphrag query \
    --root . \
    --method global \
    --query "{global_query}"




creating llm client with {'api_key': 'REDACTED,len=6', 'type': "openai_chat", 'encoding_model': 'cl100k_base', 'model': 'qwen2.5:3b', 'max_tokens': 4000, 'temperature': 0.0, 'top_p': 1.0, 'n': 1, 'frequency_penalty': 0.0, 'presence_penalty': 0.0, 'request_timeout': 180.0, 'api_base': 'http://localhost:11434/v1', 'api_version': None, 'organization': None, 'proxy': None, 'audience': None, 'deployment_name': None, 'model_supports_json': False, 'tokens_per_minute': 0, 'requests_per_minute': 0, 'max_retries': 10, 'max_retry_wait': 10.0, 'sleep_on_rate_limit_recommendation': True, 'concurrent_requests': 1, 'responses': None}

SUCCESS: Global Search Response:
According to Analyst 1's report, the two most frequent 4M (Man, Machine, Method, Measurement) root causes in the Quality Control Incident reports are 'Worker Torque Setting Error' and 'Defect Type: Low Tone Completion', which account for a significant portion of incidents [Data: Reports (3, 1)+more]. These findings suggest that address

**관찰 포인트**
- Local Search는 벡터RAG와 비슷하게 특정 엔티티 주변 정보를 답함
- Global Search는 커뮤니티 요약을 종합해 전체 패턴을 답함 — 벡터RAG로는 흉내내기 어려운 영역


## 5. Advanced 벡터RAG vs GraphRAG 동일 질의 비교

NB-05에서 구성한 Hybrid+Reranker 파이프라인과 GraphRAG의 응답을 나란히 비교합니다.


In [ ]:
# Hybrid+Reranker retriever 재구성 (새 세션이라면 다시 실행 필요)
from langchain_community.retrievers import BM25Retriever
from langchain_community.vectorstores import Chroma as LC_Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers import EnsembleRetriever   # ← langchain.retrievers 가 아님
from langchain_core.documents import Document
from konlpy.tag import Okt
from sentence_transformers import CrossEncoder

okt = Okt()
lc_documents = [Document(page_content=r["report_text"], metadata={"report_id": r["report_id"]}) for r in reports]

bm25_retriever = BM25Retriever.from_documents(lc_documents, preprocess_func=okt.morphs)
bm25_retriever.k = 10

lc_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
dense_vectorstore = LC_Chroma.from_documents(lc_documents, lc_embeddings, collection_name="nb06_compare")
dense_retriever = dense_vectorstore.as_retriever(search_kwargs={"k": 10})

hybrid_retriever = EnsembleRetriever(retrievers=[bm25_retriever, dense_retriever], weights=[0.4, 0.6])
reranker_model = CrossEncoder("BAAI/bge-reranker-base")

def advanced_vector_rag_answer(query, llm_ask_fn, top_k=3):
    candidates = hybrid_retriever.invoke(query)
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = reranker_model.predict(pairs)
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)[:top_k]
    context = "\n".join([d.page_content for d, _ in ranked])
    prompt = f"다음 정보를 참고해서 질문에 답하세요.\n\n[참고정보]\n{context}\n\n[질문]\n{query}"
    return llm_ask_fn(prompt), [d.metadata["report_id"] for d, _ in ranked]

print("Advanced 벡터RAG 파이프라인 재구성 완료")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Advanced 벡터RAG 파이프라인 재구성 완료


In [ ]:
import requests

def ask_ollama(prompt, model="qwen2.5:3b", temperature=0.2):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": model, "prompt": prompt, "stream": False, "options": {"temperature": temperature}},
    )
    return response.json()["response"]

comparison_queries = [
    "베어링 마모가 치수불량까지 이어지는 경로는 어떻게 되나요?",
#    "3호기 CNC 가공기의 최근 불량 유형은 무엇인가요?",
]

for q in comparison_queries:
    print(f"\n{'='*30}\n질의: {q}\n{'='*30}")

    vector_answer, used_ids = advanced_vector_rag_answer(q, ask_ollama)
    print(f"\n[Advanced 벡터RAG 응답] (참고: {used_ids})")
    print(vector_answer)

    print(f"\n[GraphRAG 응답] - 아래 셀에서 별도로 query 명령 실행 결과를 비교하세요")


- "3호기 최근 불량 유형은?" 같은 단순 사실 조회형 질의는 두 방식 모두 비슷한 수준의 답변 가능
- "베어링 마모가 치수불량까지 이어지는 경로" 같은 원인 연쇄 추론형 질의는 GraphRAG가 더 구체적인 경로를 제시하는 경향
- "가장 빈번한 4M 원인 분류" 같은 전체 패턴형 질의는 GraphRAG의 Global Search가 명확히 우세 — 벡터RAG는 Top-K 문서만 보므로 전체 통계를 낼 수 없음
